# Data Collection

In [1]:
import pandas as pd
import time, random
years = list(range(1975, 2026))
all_dfs = []
for year in years:
    url = f'https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html'
    df = pd.read_html(url)[0]
    df['Season'] = year
    all_dfs.append(df)
    time.sleep(random.uniform(2, 5))
final_df = pd.concat(all_dfs, ignore_index=True)
final_df.to_csv('../data/combined.csv', index=False)

import pandas as pd
df = pd.read_csv('../data/combined.csv')
print('Loaded combined.csv - shape:', df.shape)
print('Seasons:', df['Season'].min(), '-', df['Season'].max())
df.head(3)

Loaded combined.csv - shape: (26292, 32)
Seasons: 1975 - 2025


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,PTS,Awards,Season,3P,3PA,3P%,2P,2PA,2P%,eFG%
0,1.0,Bob McAdoo,23.0,BUF,C,82.0,NaN,43.2,13.4,26.1,...,34.5,"MVP-1,AS,NBA1",1975,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,Rick Barry,30.0,GSW,SF,80.0,NaN,40.4,12.9,27.7,...,30.6,"MVP-4,AS,NBA1",1975,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,Kareem Abdul-Jabbar,27.0,MIL,C,65.0,NaN,42.3,12.5,24.4,...,30.0,"MVP-5,AS,DEF1",1975,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Player Bio Data

In [2]:
import pandas as pd
import time

def height_to_inches(ht):
    # '6-7' -> 79
    try:
        feet, inches = str(ht).split('-')
        return int(feet) * 12 + int(inches)
    except:
        return None

letters = 'abcdefghijklmnopqrstuvwxyz'
bio_frames = []
for letter in letters:
    url = f'https://www.basketball-reference.com/players/{letter}/'
    try:
        table = pd.read_html(url)[0]
        bio_frames.append(table)
        print(f'[{letter}] {len(table)} players')
    except Exception as e:
        print(f'[{letter}] skipped: {e}')
    time.sleep(3)

bios = pd.concat(bio_frames, ignore_index=True)
# Hall-of-Fame players have a trailing '*' - strip it so names match combined.csv
bios['Player'] = bios['Player'].astype(str).str.replace('*', '', regex=False).str.strip()
bios['height_in'] = bios['Ht'].apply(height_to_inches)
bios['weight_lb'] = pd.to_numeric(bios['Wt'], errors='coerce')
bios = bios[['Player', 'height_in', 'weight_lb']].dropna(subset=['height_in'])
bios = bios.drop_duplicates(subset='Player', keep='first').reset_index(drop=True)
bios.to_csv('../data/player_bios.csv', index=False)

[a] 179 players
[b] 517 players
[c] 342 players
[d] 267 players
[e] 119 players
[f] 164 players
[g] 271 players
[h] 382 players
[i] 29 players
[j] 270 players
[k] 188 players
[l] 212 players
[m] 509 players
[n] 115 players
[o] 101 players
[p] 245 players
[q] 10 players
[r] 274 players
[s] 472 players
[t] 215 players
[u] 12 players
[v] 61 players
[w] 418 players
[x] 0 players
[y] 23 players
[z] 21 players


/tmp/ipykernel_143553/480436149.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bios = pd.concat(bio_frames, ignore_index=True)


In [3]:
# Check match rate vs the main dataset
main_players = pd.read_csv('../data/combined.csv')['Player'].dropna().unique()
matched = bios['Player'].isin(main_players).sum()
print(f'Bio rows: {len(bios)}')
print(f'Unique players in combined.csv: {len(main_players)}')
print(f'Matched players: {matched}')

if matched < len(main_players) / 2:
    print('WARNING: fewer than half the players matched - bio merge will use medians for most rows')

Bio rows: 5367
Unique players in combined.csv: 3953
Matched players: 3952
